In [2]:
import pandas as pd
import geopandas as gpd
import altair as alt
import json
from shapely.geometry import shape
from shapely import wkt
from shapely.ops import linemerge, substring, unary_union
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [3]:
neigh = pd.read_csv("Data/Neigh/neigh.csv")

neigh["geometry"] = neigh["geom"].apply(wkt.loads)

neigh = gpd.GeoDataFrame(
    neigh,
    geometry="geometry",
    crs="EPSG:4326"
)
neigh = neigh[["name", "geometry"]]
neigh["geometry"] = neigh.buffer(0)

neigh_layer = alt.Chart(neigh).mark_geoshape(
    stroke="black",
    fill="lightblue"
).encode(tooltip=['name'])



In [ ]:
nodes = pd.read_csv("Nodes/BUS.csv")
nodes['geometry']= nodes['geometry'].apply(wkt.loads)
nodes = gpd.GeoDataFrame(nodes, geometry='geometry', crs="EPSG:4326")

3145

In [ ]:
with open('data/Bus/AMB2.json') as f:
    data = json.load(f)


traj_amb = []

for collection in data:
    for feature in collection["features"]:
        props = feature["properties"]
        geom = feature["geometry"]
        
        if geom["type"] != "Point":
            traj_amb.append({
                "linia": props["LINEA"],
                "sentit": props["SENTIDO"],
                "geometry": shape(geom)
            })

geo_df_traj_amb = gpd.GeoDataFrame(traj_amb, geometry="geometry", crs="EPSG:4326")
#geo_df_traj_amb = (geo_df_traj_amb.groupby("linia", as_index=False).agg(geometry=("geometry", lambda s: unary_union(list(s)))))

geo_df_traj_amb = gpd.GeoDataFrame(geo_df_traj_amb,geometry="geometry",crs="EPSG:4326")

buffered = neigh.copy()
buffered.to_crs('EPSG:25831', inplace=True)
buffered["geometry"] = buffered.geometry.buffer(2500)
boundary = buffered.union_all()
geo_df_traj_amb.to_crs('EPSG:25831', inplace=True)
geo_df_traj_amb["inside_ratio"] = (
    geo_df_traj_amb.geometry.intersection(boundary).length
    / geo_df_traj_amb.geometry.length
)

geo_df_traj_amb = geo_df_traj_amb[
    geo_df_traj_amb["inside_ratio"] >= 0.60   # keep lines with >=20% inside
]


geo_df_traj_amb.to_crs('EPSG:4326', inplace=True)
geo_df_traj_amb.sort_values(by='inside_ratio', inplace=True)
geo_df_traj_amb = geo_df_traj_amb[geo_df_traj_amb['linia'].isin(nodes['linia'].unique())]
geo_df_traj_amb.sort_values(by='inside_ratio', inplace=True)
geo_df_traj_amb

In [ ]:
nodes[nodes['stop_id'] == 2275]

In [ ]:
geo_df_traj_amb[geo_df_traj_amb['linia'] == '78']

In [ ]:
lines = alt.Chart(geo_df_traj_amb[geo_df_traj_amb['linia'] == '78']).mark_geoshape(
    filled=False, 
    strokeWidth=4
).encode(
    color=alt.Color('tram:N', legend=alt.Legend(title="Metro Segments")),
    tooltip=['tram:N']
)

points = alt.Chart(nodes[nodes['linia'] == '78']).mark_geoshape(size=0, color='red').encode(tooltip=['name:N'])
plot = (lines + points).project('mercator').properties(width=800, height=600)
plot
    

In [ ]:
nodes[nodes['linia'] == '78'].sort_values(by='stop_id')

In [ ]:
def break_trajectory(trajectory_df, stops):

    line = trajectory_df.geometry.iloc[0]
    #line = trajectory_df[trajectory_df['route_id'] == 'L6'].geometry.iloc[0]

    flattened_line = unary_union(line) 
    merged_line = linemerge(flattened_line)

    if merged_line.geom_type == 'MultiLineString':
        line_s = max(merged_line.geoms, key=lambda x: x.length)
    else:
        line_s = merged_line

    stop_list = []
    for _, row in stops.iterrows():
        dist = line_s.project(row.geometry)
        stop_list.append({'name': row['name'], 'id': row['id'], 'dist': dist})
    sorted_stops = sorted(stop_list, key=lambda x: x['dist'])

    # 4. Create segments
    segments_data = []
    for i in range(len(sorted_stops) - 1):
        origin = sorted_stops[i]
        destination = sorted_stops[i+1]
        
        # Check for zero length (stops at the same location)
        if abs(destination['dist'] - origin['dist']) < 1e-7:
            print(f"Skipping zero-length segment between {origin['name']} and {destination['name']}")
            continue
            
        seg_geom = substring(line_s, origin['dist'], destination['dist'])
        
        segments_data.append({
            'origen': origin['id'],
            'dest': destination['id'],
            'tram': f"{origin['name']} - {destination['name']}",
            'linia': trajectory_df['linia'].iloc[0],
            'type' : 'Bus',
            'geometry': seg_geom,
        })

    print(f"Generated segments: {len(segments_data)}")

    segments_gdf = gpd.GeoDataFrame(segments_data, crs="EPSG:4326",geometry='geometry')

    return segments_gdf



In [ ]:
geo_df_traj_amb[geo_df_traj_amb['linia'] == '78']

In [ ]:
nodes[nodes['linia'] == '78']

In [117]:
b78 = nodes[nodes['linia'] == '78']
b78[b78['name'].str.contains('Major - Pl')]

,id,stop_id,name,linia,stop_type,geometry
13,B-78-2275,2275,Major - Pl Constitucio,78,Bus,POINT (2.05493 41.36687)


In [118]:
bus_edges = pd.DataFrame()
for route in geo_df_traj_amb['linia'].unique():
    print(f"Processing route: {route}")

    segments_gdf = break_trajectory(geo_df_traj_amb[(geo_df_traj_amb['linia'] == route) & (geo_df_traj_amb['sentit'] == 'Sentido Pl dels Paisos Catalans')], nodes[nodes['linia'] == route])
    bus_edges = pd.concat([bus_edges, segments_gdf], ignore_index=True)
bus_edges



Processing route: PR3
Processing route: 78
[{'name': 'Ctra BV 2001- Benzinera', 'id': 'B-78-3433', 'dist': 0.0}, {'name': 'Ctra BV 2001- Benzinera', 'id': 'B-78-3429', 'dist': 0.0}, {'name': "Major - Riera d'en Nofre", 'id': 'B-78-3428', 'dist': 0.0}, {'name': "Rambla Josep M Jujol - Baltasar d'Espanya", 'id': 'B-78-2274', 'dist': 0.0}, {'name': 'Major - Pl Constitucio', 'id': 'B-78-2275', 'dist': 0.0}, {'name': 'Major - Montjuic', 'id': 'B-78-2276', 'dist': 0.0}, {'name': 'Av Barcelona - Major', 'id': 'B-78-2667', 'dist': 0.0}, {'name': 'Av Barcelona - Rius i Taulet', 'id': 'B-78-2248', 'dist': 0.0}, {'name': 'Av Barcelona - Tambor del Bruc', 'id': 'B-78-904', 'dist': 0.0}, {'name': 'Fructuos Gelabert - Jacint Verdaguer', 'id': 'B-78-1739', 'dist': 0.0}, {'name': 'Av Barcelona - Mare de Deu de la Merce', 'id': 'B-78-1657', 'dist': 0.0}, {'name': 'Jacint Verdaguer  -  Josep Trueta', 'id': 'B-78-3135', 'dist': 0.0}, {'name': 'Jacint Verdaguer - de la TV3', 'id': 'B-78-1633', 'dist': 0.0

,origen,dest,tram,linia,type,geometry
0,B-78-1485,B-78-748,St Marti de l'Erm - Zona Esportiva Fontsanta -...,78,Bus,"LINESTRING (2.08393 41.37254, 2.08393 41.37254)"
1,B-78-748,B-78-1411,Av de Cornella - Pl Sardana - Av de Cornella -...,78,Bus,"LINESTRING (2.08393 41.37254, 2.0849 41.37354)"
2,B-78-1411,B-78-1460,Av de Cornella - St Francesc Xavier - Av de Co...,78,Bus,"LINESTRING (2.0849 41.37354, 2.08612 41.3748, ..."
3,B-78-1460,B-78-1461,Av de Cornella - 8 de Marc - Av de Cornella - ...,78,Bus,"LINESTRING (2.08618 41.37484, 2.08618 41.37484..."
4,B-78-1461,B-78-1459,Av de Cornella - Pont d'Esplugues - Av dels Pa...,78,Bus,"LINESTRING (2.08886 41.37683, 2.08886 41.37683..."
5,B-78-1459,B-78-1746,Av dels Paisos Catalans - Laurea Miro - Av del...,78,Bus,"LINESTRING (2.09148 41.37749, 2.09182 41.37778..."
6,B-78-1746,B-78-1458,Av dels Paisos Catalans - Laurea Miro - Av del...,78,Bus,"LINESTRING (2.09189 41.37783, 2.09189 41.37783..."
7,B-78-1458,B-78-1747,Av dels Paisos Catalans - David Carreras i Tri...,78,Bus,"LINESTRING (2.0946 41.37943, 2.0949 41.37961, ..."
8,B-78-1747,B-78-3751,Av dels Paisos Catalans - Direccio Gral d'Espo...,78,Bus,"LINESTRING (2.09496 41.37965, 2.09751 41.38184)"
9,B-78-3751,B-78-3725,Juan de la Cierva - Av Jacint Esteva Fontanet ...,78,Bus,"LINESTRING (2.09751 41.38184, 2.09794 41.38221)"


In [105]:
lines = alt.Chart(bus_edges[bus_edges['linia'] == '78']).mark_geoshape(
    filled=False, 
    strokeWidth=4
).encode(
    color=alt.Color('tram:N', legend=alt.Legend(title="Metro Segments")),
    tooltip=['tram:N']
)

points = alt.Chart(nodes[nodes['linia'] == '78']).mark_geoshape(size=0, color='red').encode(tooltip=['name:N'])
plot = (lines + points).project('mercator').properties(width=800, height=600).interactive()
plot
    

alt.LayerChart(...)